# sliding window PSI

# minimal version vis.

In [1]:
#!/usr/bin/env python3
"""
Complete PSI Pipeline — pair-level only
========================================
Sections
--------
  1. Configuration
  2. Utility functions
  3. PSI computation  (multitaper DPSS, sign: positive = ACC→vmPFC)
  4. Regression       (z-scored single-variable, one model per regressor)
  5. Permutation      (subject-constrained sign-flip, BH-FDR across time)
  6. Main analysis loop (per subject)
  7. Group-level aggregation
  8. Visualization

Sign convention
---------------
  cross_spectrum(f) = conj(ACC_fft(f)) × vmPFC_fft(f)
  PSI = Im( Σ_f  cs(f) × conj(cs(f+δf)) )
  Positive PSI → ACC phase leads vmPFC → ACC→vmPFC directed influence.
"""

import os
import warnings
import numpy as np
import pandas as pd
import mne
from mne_connectivity import seed_target_indices
from scipy.signal.windows import dpss as dpss_windows
from scipy.signal import detrend
from scipy.fft import rfft, rfftfreq
from tqdm import tqdm
import matplotlib as mpl
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────────────────────────────────────
#  1.  CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────

# Paths
IN_ROI1_PATH     = "/root/autodl-tmp/seeg_roi_1246/acc/"       # ACC epochs
IN_ROI2_PATH     = "/root/autodl-tmp/seeg_roi_1246/vmpfc/"     # vmPFC epochs
MODEL_DATA_PATH  = "/root/results/bhm_final_fix_beta_26/weighted/"
OUTPUT_ROOT      = "/root/results/sliding_window_beta_psi/"

os.makedirs(OUTPUT_ROOT, exist_ok=True)

# Subjects
SUBJECTS = [1, 2, 3, 4, 6, 7, 8, 9, 10, 11, 13, 15, 16, 19, 20, 21, 22, 24, 25]

# Epoch time range (crop before PSI to save memory)
ANALYSIS_TMIN, ANALYSIS_TMAX = -1.4, 0.2

# Sliding window parameters
DECISION_TMIN, DECISION_TMAX = -1.0, 0.1   # window centres kept in this range
WIN_LEN_SEC  = 0.30                          # window length in seconds
STEP_SEC     = 0.02                          # step between windows

# PSI frequency band (beta)
FMIN, FMAX   = 13.0, 30.0
MT_BANDWIDTH = 4.0     # multitaper half-bandwidth in Hz (controls spectral smoothing)

# Regression / permutation
MIN_TRIALS_REQUIRED = 10
N_PERMUTATIONS      = 5000
ALPHA               = 0.05

# ─────────────────────────────────────────────────────────────────────────────
#  2.  UTILITY FUNCTIONS
# ─────────────────────────────────────────────────────────────────────────────

def zscore_safe(x):
    """Z-score a 1D array; return array of NaN if zero variance."""
    x = np.asarray(x, dtype=float)
    sd = np.nanstd(x, ddof=0)
    if sd == 0 or np.isnan(sd):
        return np.full_like(x, np.nan)
    return (x - np.nanmean(x)) / sd


def align_epochs(ep1, ep2):
    """Remove '99' tags, align epoch counts."""
    ep1 = ep1[[t for t in ep1.event_id if "99" not in t]]
    ep2 = ep2[[t for t in ep2.event_id if "99" not in t]]
    n   = min(len(ep1), len(ep2))
    return ep1[:n], ep2[:n]


def build_behavior_df(subject_id, model_data_path):
    """
    Load game metrics and compute:
      reward : ΔQ = Q_chosen − mean(Q_unchosen)    z-scored within subject
      info   : I_transformed_chosen (absolute)      z-scored within subject

    Returns DataFrame or None.
    """
    path = os.path.join(model_data_path, f"{subject_id}_game_metrics.csv")
    if not os.path.exists(path):
        return None

    df   = pd.read_csv(path)
    rows = []
    for ep in range(len(df)):
        gd = df[df["Game"] == ep + 1]
        if gd.empty:
            continue
        try:
            chosen = int(gd["Chosen_Deck"].iloc[0])
            q_cols = [f"Q_Deck_{d}" for d in range(1, 4)]

            reward = (gd[f"Q_Deck_{chosen}"].iloc[0]
                      - gd[q_cols].mean(axis=1).iloc[0])

            # Prefer I_transformed over Weighted_I_transformed if available
            i_col = (f"I_transformed_Deck_{chosen}"
                     if f"I_transformed_Deck_{chosen}" in gd.columns
                     else f"Weighted_I_transformed_Deck_{chosen}")
            info = gd[i_col].iloc[0]

            rows.append({"trial_index": ep, "reward": reward, "info": info})
        except Exception:
            continue

    if not rows:
        return None

    out = pd.DataFrame(rows)
    # z-score within subject — must happen here so regressors are comparable
    out["reward_z"] = zscore_safe(out["reward"].values)
    out["info_z"]   = zscore_safe(out["info"].values)
    return out


# ─────────────────────────────────────────────────────────────────────────────
#  3.  PSI COMPUTATION
# ─────────────────────────────────────────────────────────────────────────────

def _build_windows(times, win_len_sec, step_sec, tmin_keep, tmax_keep):
    """Return (centres, start_samples, end_samples)."""
    dt        = times[1] - times[0]
    sfreq     = 1.0 / dt
    win_samp  = int(round(win_len_sec * sfreq))
    step_samp = max(1, int(round(step_sec * sfreq)))
    half      = win_samp // 2

    centres, starts, ends = [], [], []
    for c in range(half, len(times) - half, step_samp):
        tc = times[c]
        if tc < tmin_keep or tc > tmax_keep:
            continue
        s, e = c - half, c - half + win_samp
        if e > len(times):
            break
        centres.append(tc)
        starts.append(s)
        ends.append(e)

    return np.array(centres), np.array(starts, dtype=int), np.array(ends, dtype=int)


def _compute_psi_one_pair_one_window(x_seg, y_seg, sfreq, fmin, fmax, bandwidth):
    """
    PSI for one ACC–vmPFC pair in one time window.

    cross_spectrum = conj(x_fft) × y_fft
    PSI = Im( Σ_f  cs(f) × conj(cs(f+1)) )
    Positive PSI → x (ACC) phase leads y (vmPFC) → ACC→vmPFC.

    Parameters
    ----------
    x_seg : 1D array  (ACC channel segment)
    y_seg : 1D array  (vmPFC channel segment)
    sfreq : float     (sampling frequency, Hz)
    fmin, fmax : float (frequency band, Hz)
    bandwidth  : float (multitaper half-bandwidth, Hz)

    Returns
    -------
    psi : float  (NaN if insufficient data)
    """
    n = len(x_seg)
    if n < 4 or np.all(x_seg == 0) or np.all(y_seg == 0):
        return np.nan

    # Linear detrend
    x_seg = detrend(x_seg, type="linear")
    y_seg = detrend(y_seg, type="linear")

    # DPSS tapers
    nw      = max(1.5, bandwidth * n / sfreq / 2.0)   # time–half-bandwidth product
    n_tap   = max(1, int(2 * nw) - 1)
    try:
        tapers = dpss_windows(n, nw, Kmax=n_tap)
        if tapers.ndim == 1:
            tapers = tapers[np.newaxis, :]
    except Exception:
        tapers = np.array([np.ones(n) / np.sqrt(n)])

    # Frequency mask
    freqs = rfftfreq(n, 1.0 / sfreq)
    fmask = (freqs >= fmin) & (freqs <= fmax)
    fi    = np.where(fmask)[0]
    if len(fi) < 2:
        return np.nan

    # Accumulate cross-spectrum over tapers
    psi_acc = 0.0
    for tap in tapers:
        xf  = rfft(x_seg * tap)
        yf  = rfft(y_seg * tap)
        cs  = np.conj(xf) * yf                       # seed* × target
        # Im(cs(f) × conj(cs(f+1)))  → positive when x leads y
        psi_acc += np.imag(
            np.sum(cs[fi[:-1]] * np.conj(cs[fi[1:]]))
        )

    return float(psi_acc / len(tapers))


def compute_sliding_window_psi_all_trials(
    data_seed,          # (n_trials, n_seed_ch, n_times)
    data_target,        # (n_trials, n_target_ch, n_times)
    indices,            # output of seed_target_indices
    sfreq,
    starts, ends,       # window sample boundaries
    fmin, fmax,
    bandwidth,
):
    """
    Return PSI array of shape (n_trials, n_pairs, n_windows).
    Positive PSI: seed leads target.
    """
    n_trials  = data_seed.shape[0]
    n_pairs   = len(indices[0])
    n_windows = len(starts)

    psi_out = np.full((n_trials, n_pairs, n_windows), np.nan, dtype=float)

    for w_idx in range(n_windows):
        s, e = int(starts[w_idx]), int(ends[w_idx])
        for tr in range(n_trials):
            for p_idx in range(n_pairs):
                si = indices[0][p_idx]           # seed channel index
                ti = indices[1][p_idx]           # target channel index
                # (indices[1] already points into the target array)
                x_seg = data_seed  [tr, si, s:e]
                y_seg = data_target[tr, ti, s:e]
                psi_out[tr, p_idx, w_idx] = _compute_psi_one_pair_one_window(
                    x_seg, y_seg, sfreq, fmin, fmax, bandwidth
                )

    return psi_out


# ─────────────────────────────────────────────────────────────────────────────
#  4.  REGRESSION
# ─────────────────────────────────────────────────────────────────────────────

def timepoint_regression_single(y_mat, x_z):
    """
    For each window: fit  PSI ~ 1 + x_z  and return the slope.

    Parameters
    ----------
    y_mat : (n_trials, n_windows) — PSI values
    x_z   : (n_trials,) — z-scored regressor

    Returns
    -------
    beta : (n_windows,) — slope coefficients
    """
    y_mat = np.asarray(y_mat, float)
    x_z   = np.asarray(x_z,   float)
    n_trials, n_windows = y_mat.shape
    beta = np.full(n_windows, np.nan)

    for t in range(n_windows):
        y     = y_mat[:, t]
        valid = np.isfinite(y) & np.isfinite(x_z)
        if valid.sum() < 5:
            continue
        xv, yv = x_z[valid], y[valid]
        if np.nanstd(xv) == 0:
            continue
        X = np.column_stack([np.ones(valid.sum()), xv])
        try:
            coef, _, _, _ = np.linalg.lstsq(X, yv, rcond=None)
            beta[t] = coef[1]
        except Exception:
            pass

    return beta


# ─────────────────────────────────────────────────────────────────────────────
#  5.  PERMUTATION (pair-level, subject-constrained sign-flip + BH-FDR)
# ─────────────────────────────────────────────────────────────────────────────

def _signflip_stat(subject_groups):
    """Mean of per-subject means (equal-weight across subjects)."""
    subj_means = [np.nanmean(v) for v in subject_groups.values() if len(v) > 0]
    return float(np.nanmean(subj_means)) if subj_means else np.nan


def signflip_test_one_time(df_t, subject_col, value_col, n_perm, rng):
    """
    Subject-constrained sign-flip permutation at a single time point.
    All pairs of the same subject are flipped together.
    Returns (observed_stat, p_value, n_subjects).
    """
    data = df_t[[subject_col, value_col]].dropna()
    groups = {
        sid: grp[value_col].to_numpy(float)
        for sid, grp in data.groupby(subject_col)
        if grp[value_col].notna().any()
    }
    if not groups:
        return np.nan, np.nan, 0

    obs  = _signflip_stat(groups)
    sids = list(groups.keys())

    null = np.empty(n_perm)
    for i in range(n_perm):
        perm = {
            sid: groups[sid] * rng.choice([-1, 1])
            for sid in sids
        }
        null[i] = _signflip_stat(perm)

    p = (np.sum(np.abs(null) >= abs(obs)) + 1) / (n_perm + 1)
    return obs, p, len(groups)


def run_time_resolved_signflip(df_pairs, subject_col, time_col, value_col,
                                n_perm, seed=42):
    """
    Run sign-flip permutation at every unique time point.
    Returns DataFrame with columns [time, observed_stat, p_value, n_subjects].
    """
    times = np.sort(df_pairs[time_col].dropna().unique())
    rng   = np.random.default_rng(seed)
    rows  = []
    for i, t in enumerate(tqdm(times, desc=f"  Sign-flip perm ({value_col})")):
        df_t   = df_pairs[df_pairs[time_col] == t]
        obs, p, ns = signflip_test_one_time(
            df_t, subject_col, value_col,
            n_perm, np.random.default_rng(seed + i)
        )
        rows.append({"time": float(t), "observed_stat": obs,
                     "p_value": p, "n_subjects": ns})
    return pd.DataFrame(rows)


def add_fdr_bh(df, p_col="p_value", out_col="p_fdr_bh"):
    """Benjamini–Hochberg FDR correction."""
    out   = df.copy()
    pvals = out[p_col].to_numpy(float)
    valid = np.isfinite(pvals)
    qvals = np.full_like(pvals, np.nan)
    if valid.sum() > 0:
        p     = pvals[valid]
        m     = len(p)
        order = np.argsort(p)
        q     = p[order] * m / np.arange(1, m + 1)
        q     = np.minimum.accumulate(q[::-1])[::-1]
        q     = np.clip(q, 0, 1)
        q_out = np.empty_like(q)
        q_out[order] = q
        qvals[valid]  = q_out
    out[out_col] = qvals
    return out


# ─────────────────────────────────────────────────────────────────────────────
#  6.  MAIN ANALYSIS LOOP
# ─────────────────────────────────────────────────────────────────────────────

pairwise_rows  = []     # one row per (subject, pair, time)
subject_betas  = {"reward": [], "info": []}
subject_ids_kept = []
times_final    = None

for subj in SUBJECTS:
    print(f"\n{'─'*60}\nSubject {subj}")
    try:
        # Load epochs
        ep1 = mne.read_epochs_eeglab(f"{IN_ROI1_PATH}{subj}.set", verbose=False)
        ep2 = mne.read_epochs_eeglab(f"{IN_ROI2_PATH}{subj}.set", verbose=False)
        ep1, ep2 = align_epochs(ep1, ep2)

        if len(ep1) == 0:
            print("  No valid epochs, skip."); continue

        ep1 = ep1.crop(ANALYSIS_TMIN, ANALYSIS_TMAX)
        ep2 = ep2.crop(ANALYSIS_TMIN, ANALYSIS_TMAX)

        sfreq = float(ep1.info["sfreq"])
        times = ep1.times

        data1 = ep1.get_data(copy=False)    # (n_ep, n_ch1, n_t)
        data2 = ep2.get_data(copy=False)    # (n_ep, n_ch2, n_t)

        n_ep = min(data1.shape[0], data2.shape[0])
        data1, data2 = data1[:n_ep], data2[:n_ep]

        n_ch1 = data1.shape[1]
        n_ch2 = data2.shape[1]
        ch_names1 = ep1.ch_names
        ch_names2 = ep2.ch_names

        # Pair indices: every ACC × vmPFC combination
        roi1_idx = np.arange(n_ch1)
        roi2_idx = np.arange(n_ch2)
        # seed_target_indices expects flat index arrays (not combined)
        seed_idx   = np.repeat(roi1_idx, n_ch2)
        target_idx = np.tile(roi2_idx, n_ch1)
        indices    = (seed_idx, target_idx)
        n_pairs    = len(seed_idx)

        # Build sliding windows
        centres, starts, ends = _build_windows(
            times, WIN_LEN_SEC, STEP_SEC, DECISION_TMIN, DECISION_TMAX
        )
        if len(centres) == 0:
            print("  No windows in decision period, skip."); continue

        print(f"  Epochs: {n_ep} | ACC ch: {n_ch1} | vmPFC ch: {n_ch2} "
              f"| Pairs: {n_pairs} | Windows: {len(centres)}")

        # Compute PSI for all trials × pairs × windows
        print("  Computing sliding-window PSI…")
        psi_arr = compute_sliding_window_psi_all_trials(
            data_seed   = data1,
            data_target = data2,
            indices     = indices,
            sfreq       = sfreq,
            starts      = starts,
            ends        = ends,
            fmin        = FMIN,
            fmax        = FMAX,
            bandwidth   = MT_BANDWIDTH,
        )   # (n_trials, n_pairs, n_windows)

        # Load behaviour + z-score regressors
        behav = build_behavior_df(subj, MODEL_DATA_PATH)
        if behav is None:
            print("  No behaviour data, skip."); continue

        behav = behav[behav["trial_index"] < n_ep].reset_index(drop=True)
        n_trials = min(len(behav), psi_arr.shape[0])
        if n_trials < MIN_TRIALS_REQUIRED:
            print(f"  Too few trials ({n_trials}), skip."); continue

        behav   = behav.iloc[:n_trials]
        psi_arr = psi_arr[:n_trials]

        reward_z = behav["reward_z"].to_numpy(float)
        info_z   = behav["info_z"].to_numpy(float)

        # Regression: one model per regressor, z-scored predictors
        pair_beta_r, pair_beta_i = [], []

        for p_idx in tqdm(range(n_pairs), desc="  Regression", leave=False):
            y_mat = psi_arr[:, p_idx, :]    # (n_trials, n_windows)

            beta_r = timepoint_regression_single(y_mat, reward_z)
            beta_i = timepoint_regression_single(y_mat, info_z)

            if np.all(np.isnan(beta_r)) and np.all(np.isnan(beta_i)):
                continue

            pair_beta_r.append(beta_r)
            pair_beta_i.append(beta_i)

            ch1 = ch_names1[seed_idx[p_idx]]
            ch2 = ch_names2[target_idx[p_idx]]

            for w_idx, t_val in enumerate(centres):
                pairwise_rows.append({
                    "subject":       str(subj),
                    "seed_ch":       ch1,
                    "target_ch":     ch2,
                    "time":          float(t_val),
                    "beta_reward":   float(beta_r[w_idx]),
                    "beta_info":     float(beta_i[w_idx]),
                })

        if not pair_beta_r:
            print("  No valid pairs, skip."); continue

        pair_beta_r = np.array(pair_beta_r, dtype=float)   # (n_valid_pairs, n_win)
        pair_beta_i = np.array(pair_beta_i, dtype=float)

        # Subject-level summary: mean across pairs
        subj_mean_r = np.nanmean(pair_beta_r, axis=0)
        subj_mean_i = np.nanmean(pair_beta_i, axis=0)

        subject_betas["reward"].append(subj_mean_r)
        subject_betas["info"].append(subj_mean_i)
        subject_ids_kept.append(str(subj))

        # Check time axis consistency
        if times_final is None:
            times_final = centres.copy()
        elif len(times_final) != len(centres) or not np.allclose(times_final, centres):
            print("  Inconsistent time axis, drop subject.")
            subject_betas["reward"].pop()
            subject_betas["info"].pop()
            subject_ids_kept.pop()
            pairwise_rows = [r for r in pairwise_rows if r["subject"] != str(subj)]
            continue

        print(f"  Done. Kept pairs: {len(pair_beta_r)}")

    except Exception as exc:
        print(f"  FAILED: {exc}")
        continue


# ─────────────────────────────────────────────────────────────────────────────
#  7.  GROUP-LEVEL AGGREGATION
# ─────────────────────────────────────────────────────────────────────────────

if not subject_betas["reward"]:
    raise RuntimeError("No subjects completed successfully.")

subj_beta_reward = np.array(subject_betas["reward"])   # (n_subj, n_win)
subj_beta_info   = np.array(subject_betas["info"])

print(f"\n{'='*60}")
print(f"Kept subjects: {subject_ids_kept}")
print(f"Shape: {subj_beta_reward.shape}")

# Save numpy arrays
np.save(os.path.join(OUTPUT_ROOT, "times_centres.npy"),   times_final)
np.save(os.path.join(OUTPUT_ROOT, "subj_beta_reward.npy"), subj_beta_reward)
np.save(os.path.join(OUTPUT_ROOT, "subj_beta_info.npy"),   subj_beta_info)
pd.DataFrame({"subject": subject_ids_kept}).to_csv(
    os.path.join(OUTPUT_ROOT, "kept_subjects.csv"), index=False)

# Save pairwise long table
pairwise_df = pd.DataFrame(pairwise_rows)
pairwise_df.to_csv(
    os.path.join(OUTPUT_ROOT, "pairwise_beta_long.csv"), index=False)

# Split into per-regressor long tables for permutation
df_reward_pairs = (pairwise_df[["subject","time","beta_reward"]]
                   .rename(columns={"beta_reward": "value"})
                   .dropna(subset=["value"])
                   .reset_index(drop=True))

df_info_pairs = (pairwise_df[["subject","time","beta_info"]]
                 .rename(columns={"beta_info": "value"})
                 .dropna(subset=["value"])
                 .reset_index(drop=True))

# ─────────────────────────────────────────────────────────────────────────────
#  Pair-level subject-constrained sign-flip + BH-FDR
# ─────────────────────────────────────────────────────────────────────────────

print("\nPair-level permutation — reward…")
reward_perm_df = run_time_resolved_signflip(
    df_reward_pairs, "subject", "time", "value", N_PERMUTATIONS, seed=42)

print("Pair-level permutation — info…")
info_perm_df = run_time_resolved_signflip(
    df_info_pairs, "subject", "time", "value", N_PERMUTATIONS, seed=4242)

reward_perm_df = add_fdr_bh(reward_perm_df)
info_perm_df   = add_fdr_bh(info_perm_df)

reward_perm_df["sig_raw"] = reward_perm_df["p_value"]  < ALPHA
reward_perm_df["sig_fdr"] = reward_perm_df["p_fdr_bh"] < ALPHA
info_perm_df  ["sig_raw"] = info_perm_df  ["p_value"]  < ALPHA
info_perm_df  ["sig_fdr"] = info_perm_df  ["p_fdr_bh"] < ALPHA

reward_perm_df.to_csv(os.path.join(OUTPUT_ROOT, "reward_perm_results.csv"), index=False)
info_perm_df  .to_csv(os.path.join(OUTPUT_ROOT, "info_perm_results.csv"),   index=False)

print("Permutation results saved.")


# ─────────────────────────────────────────────────────────────────────────────
#  8.  VISUALIZATION
# ─────────────────────────────────────────────────────────────────────────────

mpl.rcParams.update({
    "font.family":       "Arial",
    "pdf.fonttype":      42,
    "ps.fonttype":       42,
    "svg.fonttype":      "none",
    "font.size":         9,
    "axes.labelsize":    9,
    "axes.titlesize":    9,
    "xtick.labelsize":   8,
    "ytick.labelsize":   8,
    "legend.fontsize":   7.5,
    "axes.linewidth":    0.8,
    "figure.dpi":        150,
    "savefig.dpi":       600,
    "savefig.bbox":      "tight",
})

FIG_W_IN = 88.0 / 25.4    # single-column width (Nature: 88 mm)
C_PINK   = "#D36179"
C_TEAL   = "#7BB5B7"
C_SIG    = "#BF3363"


def find_sig_segments(x, mask):
    """List of (x_start, x_end) for contiguous True runs in mask."""
    x, m = np.asarray(x, float), np.asarray(mask, bool)
    if not m.any():
        return []
    idx = np.where(m)[0]
    segs, s, p = [], idx[0], idx[0]
    for i in idx[1:]:
        if i == p + 1:
            p = i
        else:
            segs.append((x[s], x[p]))
            s = p = i
    segs.append((x[s], x[p]))
    return segs


def save_figure(fig, base_name):
    for ext in ("pdf", "svg", "png"):
        kw = {"transparent": True}
        if ext == "png":
            kw["dpi"] = 600
        fig.savefig(os.path.join(OUTPUT_ROOT, f"{base_name}.{ext}"), **kw)


def plot_psi_timecourse(
    times_all,          # all window centres (full array)
    subj_beta,          # (n_subjects, n_windows) — for SEM ribbon
    perm_df,            # output of run_time_resolved_signflip + FDR
    effect_name,        # "Reward" or "Info"
    line_color,
    tmin=-0.6, tmax=0.1,
):
    """
    Single-panel PSI timecourse figure.

    Shows
    -----
    · Ribbon    : between-subject SEM (from subj_beta)
    · Main line : pair-level observed statistic (from perm_df)
    · Bars      : raw p < .05 (thin) and FDR q < .05 (thick) significance markers
    · Ref lines : y=0 (dashed), t=0/choice onset (dotted)
    """
    # Window mask
    win = (times_all >= tmin) & (times_all <= tmax)
    t   = times_all[win]

    # Between-subject SEM
    n_s     = subj_beta.shape[0]
    s_mean  = np.nanmean(subj_beta[:, win], axis=0)
    s_sem   = np.nanstd (subj_beta[:, win], axis=0, ddof=1) / np.sqrt(n_s)

    # Pair-level permutation results
    perm = perm_df.sort_values("time").reset_index(drop=True)
    pw   = (perm["time"].to_numpy(float) >= tmin) & \
           (perm["time"].to_numpy(float) <= tmax)
    pt       = perm.loc[pw, "time"].to_numpy(float)
    stat     = perm.loc[pw, "observed_stat"].to_numpy(float)
    raw_sig  = perm.loc[pw, "sig_raw"].to_numpy(bool)
    fdr_sig  = perm.loc[pw, "sig_fdr"].to_numpy(bool)

    # ── figure ────────────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(FIG_W_IN, FIG_W_IN * 0.72))

    # SEM ribbon (between-subject variability)
    ax.fill_between(t, s_mean - s_sem, s_mean + s_sem,
                    color=line_color, alpha=0.20, linewidth=0,
                    label=f"Between-subject s.e.m. (N={n_s})", zorder=1)

    # Main line: pair-level observed statistic
    ax.plot(pt, stat, color=line_color, linewidth=2.0,
            solid_capstyle="round",
            label=f"{effect_name} β (pair-level)", zorder=3)

    # Reference lines
    ax.axhline(0, color="0.55", linewidth=0.8, linestyle="--", zorder=0)
    ax.axvline(0, color="0.55", linewidth=0.8, linestyle=":",  zorder=0,
               label="Choice onset")

    # ── significance bars below axis ─────────────────────────────────────────
    all_lo = min(np.nanmin(s_mean - s_sem), np.nanmin(stat))
    all_hi = max(np.nanmax(s_mean + s_sem), np.nanmax(stat))
    yr     = (all_hi - all_lo) if all_hi > all_lo else 1.0

    y_raw = all_lo - 0.10 * yr
    y_fdr = all_lo - 0.18 * yr

    for i, (x0, x1) in enumerate(find_sig_segments(pt, raw_sig)):
        ax.plot([x0, x1], [y_raw, y_raw], color=C_SIG, linewidth=2.0,
                solid_capstyle="butt", clip_on=False, zorder=4,
                label="$p < .05$ (uncorrected)" if i == 0 else None)

    for i, (x0, x1) in enumerate(find_sig_segments(pt, fdr_sig)):
        ax.plot([x0, x1], [y_fdr, y_fdr], color=C_SIG, linewidth=3.5,
                solid_capstyle="butt", clip_on=False, zorder=5,
                label="$q < .05$ (FDR-BH)" if i == 0 else None)

    # Right-side bar labels (outside plot area)
    ax.text(tmax + 0.015, y_raw, "$p<.05$", va="center", ha="left",
            fontsize=6.5, color=C_SIG, clip_on=False)
    ax.text(tmax + 0.015, y_fdr, "FDR",    va="center", ha="left",
            fontsize=6.5, color=C_SIG, clip_on=False)

    # ── cosmetics ─────────────────────────────────────────────────────────────
    ax.set_ylim(y_fdr - 0.08 * yr, all_hi + 0.10 * yr)
    ax.set_xlim(tmin - 0.02, tmax + 0.05)
    ax.set_xlabel("Time relative to choice onset (s)", labelpad=3)
    ax.set_ylabel("PSI β  (ACC→vmPFC, 13–30 Hz)", labelpad=3)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.yaxis.set_ticks_position("left")
    ax.xaxis.set_ticks_position("bottom")
    ax.tick_params(direction="out", length=3)

    ax.legend(loc="upper left", frameon=False,
              handlelength=1.2, borderpad=0.2, labelspacing=0.25)

    plt.tight_layout()
    fname = f"psi_{effect_name.lower()}_beta_timecourse"
    save_figure(fig, fname)
    plt.close(fig)
    print(f"  ✓ {fname}.pdf / .svg / .png")


# ── Generate figures ──────────────────────────────────────────────────────────
print("\nGenerating figures…")

plot_psi_timecourse(
    times_final, subj_beta_reward, reward_perm_df,
    "Reward", C_PINK,
)

plot_psi_timecourse(
    times_final, subj_beta_info, info_perm_df,
    "Info", C_TEAL,
)

print(f"\n✓ All outputs saved to: {OUTPUT_ROOT}")


────────────────────────────────────────────────────────────
Subject 1
  Epochs: 50 | ACC ch: 5 | vmPFC ch: 10 | Pairs: 50 | Windows: 53
  Computing sliding-window PSI…


  Done. Kept pairs: 50

────────────────────────────────────────────────────────────
Subject 2
  Epochs: 77 | ACC ch: 5 | vmPFC ch: 14 | Pairs: 70 | Windows: 53
  Computing sliding-window PSI…


  Done. Kept pairs: 70

────────────────────────────────────────────────────────────
Subject 3
  FAILED: File does not exist: "/root/autodl-tmp/seeg_roi_1246/acc/3.set"

────────────────────────────────────────────────────────────
Subject 4
  FAILED: File does not exist: "/root/autodl-tmp/seeg_roi_1246/acc/4.set"

────────────────────────────────────────────────────────────
Subject 6
  Epochs: 100 | ACC ch: 8 | vmPFC ch: 16 | Pairs: 128 | Windows: 53
  Computing sliding-window PSI…


  Done. Kept pairs: 128

────────────────────────────────────────────────────────────
Subject 7
  Epochs: 100 | ACC ch: 7 | vmPFC ch: 18 | Pairs: 126 | Windows: 53
  Computing sliding-window PSI…


  Done. Kept pairs: 126

────────────────────────────────────────────────────────────
Subject 8
  FAILED: File does not exist: "/root/autodl-tmp/seeg_roi_1246/acc/8.set"

────────────────────────────────────────────────────────────
Subject 9
  FAILED: File does not exist: "/root/autodl-tmp/seeg_roi_1246/acc/9.set"

────────────────────────────────────────────────────────────
Subject 10
  Epochs: 100 | ACC ch: 18 | vmPFC ch: 30 | Pairs: 540 | Windows: 53
  Computing sliding-window PSI…


...

Permutation results saved.

Generating figures…


findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family ['Arial'] not found. Falling back to DejaVu Sans.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Ari

  ✓ psi_reward_beta_timecourse.pdf / .svg / .png


findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font f

  ✓ psi_info_beta_timecourse.pdf / .svg / .png

✓ All outputs saved to: /root/results/sliding_window_beta_psi/
